# Session 6 · A batch pipeline, then break it

Crown Street Markets, a fictional Charlotte grocery chain with 40 stores, builds the sales table its store managers read at 7 a.m.

In [ ]:
source ~/dsba6190-live-demo-06/env.sh || echo "NOT STAGED. Run prep.ipynb first. Do not run any other cell."
WORK="${WORK:-$HOME/dsba6190-live-demo-06}"
echo "$INSTANCE   $BUCKET   $FQ"

## Step 1 · The instance, the edition, and the meter

In [ ]:
gcloud beta data-fusion instances describe "$INSTANCE" \
  --project "$PROJECT" --location us-central1 \
  --format="yaml(name,state,type,version,apiEndpoint)"

## Step 2 · The recipe is a directive list

In [ ]:
echo "gs://$BUCKET/raw/pos/pos-2026-09-24.csv"

## Step 3 · Three nodes, two edges

In [ ]:
SHAPE='(.config.stages[] | "\(.name)\t\(.plugin.type)\t\(.plugin.name)"),
       (.config.connections[] | "\(.from)\t->\t\(.to)")'

jq -r "$SHAPE" "$WORK/pipelines/01-baseline.json" | column -t -s $'\t'

## Step 4 · Deploy and run · run A

In [ ]:
python3 cdap.py deploy --endpoint "$ENDPOINT" --app pos-01-baseline \
  --file "$WORK/pipelines/01-baseline.json"

In [ ]:
RUN_A=$(python3 cdap.py start --endpoint "$ENDPOINT" --app pos-01-baseline)
echo "$RUN_A"

In [ ]:
gcloud dataproc clusters list --project "$PROJECT" --region us-central1 \
  --format="table(clusterName,status.state,config.masterConfig.machineTypeUri.basename(),config.workerConfig.numInstances,config.workerConfig.machineTypeUri.basename())"

## Step 5 · What one run provisions, and what it costs

In [ ]:
python3 cdap.py wait --endpoint "$ENDPOINT" --app pos-01-baseline --run "$RUN_A"

## Step 6 · The control number

In [ ]:
bq --project_id="$PROJECT" --quiet query --use_legacy_sql=false --nouse_cache --format=pretty \
  "SELECT COUNT(*) AS rows_loaded, ROUND(SUM(amount), 2) AS total_amount,
          COUNTIF(txn_ts IS NULL) AS null_ts
   FROM \`$FQ\`"

## Step 7 · The malformed day arrives · run B

In [ ]:
gcloud storage cp "$WORK/sample/pos-2026-09-24-dirty.csv" \
  "gs://$BUCKET/raw/pos/pos-2026-09-24.csv"

In [ ]:
tail -3 "$WORK/sample/pos-2026-09-24-dirty.csv"

In [ ]:
RUN_B=$(python3 cdap.py start --endpoint "$ENDPOINT" --app pos-01-baseline)
echo "$RUN_B"

## Step 8 · Retry, quarantine, halt

In [ ]:
python3 cdap.py wait --endpoint "$ENDPOINT" --app pos-01-baseline --run "$RUN_B"

## Step 9 · One query, two defects

In [ ]:
bq --project_id="$PROJECT" --quiet query --use_legacy_sql=false --nouse_cache --format=pretty \
  "SELECT COUNT(*) AS rows_loaded FROM \`$FQ\`"

In [ ]:
python3 cdap.py stages --endpoint "$ENDPOINT" --app pos-01-baseline \
  --run "$RUN_B" --stage Wrangler

In [ ]:
bq --project_id="$PROJECT" --quiet query --use_legacy_sql=false --nouse_cache --format=pretty \
  "SELECT txn_id, txn_ts, qty, amount FROM \`$FQ\`
   WHERE txn_id IN ('T-0900001','T-0900002','T-0900003') ORDER BY txn_id"

## Step 10 · The fix is two properties · run C

In [ ]:
diff <(jq -r "$SHAPE" "$WORK/pipelines/01-baseline.json") \
     <(jq -r "$SHAPE" "$WORK/pipelines/02-quarantine.json") \
  | grep '^>' | column -t -s $'\t'

In [ ]:
echo "gs://$BUCKET/quarantine/pos"

In [ ]:
bq --project_id="${PROJECT:?}" rm -f -t "${DATASET:?}.sales_validated"
python3 cdap.py deploy --endpoint "$ENDPOINT" --app pos-02-quarantine \
  --file "$WORK/pipelines/02-quarantine.json"

In [ ]:
RUN_C=$(python3 cdap.py start --endpoint "$ENDPOINT" --app pos-02-quarantine)
echo "$RUN_C"

## Step 11 · Field-level lineage, then the count that reconciles

In [ ]:
python3 cdap.py wait --endpoint "$ENDPOINT" --app pos-02-quarantine --run "$RUN_C"

In [ ]:
bq --project_id="$PROJECT" --quiet query --use_legacy_sql=false --nouse_cache --format=pretty \
  "SELECT COUNT(*) AS rows_loaded FROM \`$FQ\`"

In [ ]:
gcloud storage cat "gs://$BUCKET/quarantine/pos/**" | head -4

## Step 12 · Change nothing, run it again · run D

In [ ]:
RUN_D=$(python3 cdap.py start --endpoint "$ENDPOINT" --app pos-02-quarantine)
echo "$RUN_D"

In [ ]:
python3 cdap.py wait --endpoint "$ENDPOINT" --app pos-02-quarantine --run "$RUN_D"

In [ ]:
bq --project_id="$PROJECT" --quiet query --use_legacy_sql=false --nouse_cache --format=pretty \
  "SELECT COUNT(*) AS rows_loaded, COUNT(DISTINCT txn_id) AS distinct_txn_id FROM \`$FQ\`"

In [ ]:
python3 cdap.py logs --endpoint "$ENDPOINT" --app pos-02-quarantine \
  --run "$RUN_D" --grep FileAlreadyExists --tail 1

## Delete the instance

In [ ]:
gcloud beta data-fusion instances delete "${INSTANCE:?}" \
  --project "${PROJECT:?}" --location us-central1 --quiet

In [ ]:
gcloud beta data-fusion instances list --project "$PROJECT" --location us-central1

---
# After class · Teardown

In [ ]:
gcloud dataproc clusters list --project "$PROJECT" --region us-central1

In [ ]:
gcloud storage rm -r "gs://${BUCKET:?}"
bq --project_id="${PROJECT:?}" rm -r -f -d "${DATASET:?}"

In [ ]:
gcloud storage ls --project "$PROJECT"
gcloud storage buckets list --project "$PROJECT" \
  --filter="labels.cdf_instance=$INSTANCE" --format="value(name)"
bq --project_id="$PROJECT" ls